# AI-DLC Study Planner Generator

## From Co-pilot to Creator: AI-Driven Development Lifecycle Coursework

This notebook implements an AI-powered meta-software development workflow. From one business problem, it automatically generates SDLC artefacts, UML diagrams, a functional Flask API, a website with a generated image, tests, Docker configuration, and a GitHub Actions CI workflow.

The generated application is an **AI Study Planner** that helps students create a weekly study plan from course name, difficulty level, available hours, learning goal, and deadline.

## 1. Setup, Business Problem, and AI-DLC Generation

The business problem is the single source of truth. The notebook uses LLM-style prompts and deterministic fallback outputs so the coursework can be reproduced without relying on an external API key.

In [ ]:
from pathlib import Path
from textwrap import dedent
import json
import os
import time
from datetime import datetime

try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    Image = ImageDraw = ImageFont = None

try:
    import requests
except ImportError:
    requests = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

BASE = Path("generated_project")
APP = BASE / "app"
TEMPLATES = APP / "templates"
STATIC = APP / "static"
ARTIFACTS = BASE / "artifacts"
TESTS = BASE / "tests"
WORKFLOWS = BASE / ".github" / "workflows"

for path in [APP, TEMPLATES, STATIC, ARTIFACTS, TESTS, WORKFLOWS]:
    path.mkdir(parents=True, exist_ok=True)

def write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(content).strip() + "\n", encoding="utf-8")
    print(f"Saved: {path}")

def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved: {path}")

if load_dotenv is not None:
    load_dotenv(Path(".env"))

USE_AI_API = os.getenv("USE_AI_API", os.getenv("USE_OPENAI_API", "false")).lower() == "true"
AI_PROVIDER = os.getenv("AI_PROVIDER", "apifree")
APIFREE_BASE_URL = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1").rstrip("/")
APIFREE_API_KEY = os.getenv("APIFREE_API_KEY") or os.getenv("OPENAI_API_KEY")
AI_TEXT_MODEL = os.getenv("AI_TEXT_MODEL", os.getenv("OPENAI_TEXT_MODEL", "openai/gpt-5.2"))
AI_IMAGE_MODEL = os.getenv("AI_IMAGE_MODEL", os.getenv("OPENAI_IMAGE_MODEL", "openai/gpt-image-2"))
AI_MAX_TOKENS = int(os.getenv("AI_MAX_TOKENS", "4096"))
AI_IMAGE_SIZE = os.getenv("AI_IMAGE_SIZE", "2048x1152")
AI_IMAGE_QUALITY = os.getenv("AI_IMAGE_QUALITY", "high")

AI_CALL_LOG = []

def has_valid_api_key():
    return bool(
        USE_AI_API
        and APIFREE_API_KEY
        and APIFREE_API_KEY != "PASTE_YOUR_API_KEY_HERE"
        and requests is not None
    )

def clean_fenced_output(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
    return cleaned

def ai_text(prompt: str, fallback: str, label: str) -> str:
    if not has_valid_api_key():
        AI_CALL_LOG.append({"label": label, "mode": "fallback", "reason": "API disabled, missing key, or requests unavailable"})
        return dedent(fallback).strip()

    headers = {
        "Authorization": f"Bearer {APIFREE_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": AI_TEXT_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert software engineering assistant. "
                    "Generate concise, coursework-ready artefacts for an AI-DLC Flask project. "
                    "Return only the requested artefact, with no chatty preface."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "max_tokens": AI_MAX_TOKENS,
        "stream": False,
    }

    try:
        response = requests.post(f"{APIFREE_BASE_URL}/chat/completions", headers=headers, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        if data.get("error"):
            raise RuntimeError(data["error"])
        content = data["choices"][0]["message"]["content"]
        if not content or not content.strip():
            raise RuntimeError("empty model response")
        AI_CALL_LOG.append({"label": label, "mode": "api", "model": AI_TEXT_MODEL})
        return clean_fenced_output(content)
    except Exception as exc:
        AI_CALL_LOG.append({"label": label, "mode": "fallback", "reason": str(exc)[:180]})
        return dedent(fallback).strip()

business_problem = """
Students often struggle to organise independent study time across multiple modules.
They need a web-based AI Study Planner that generates a personalised weekly study plan
based on course name, difficulty level, available study hours, learning goal, and deadline.
""".strip()

prompts = {
    "problem_statement": f"Generate one concise problem statement for this business problem:\n{business_problem}",
    "personas": f"Generate 3 concise personas for this system. Use markdown and include responsibilities and needs.\nBusiness problem:\n{business_problem}",
    "requirements": f"Generate functional and non-functional requirements for a Flask API and website. Use markdown headings.\nBusiness problem:\n{business_problem}",
    "user_stories": f"Generate 4 user stories with acceptance criteria for an AI Study Planner. Use concise markdown.\nBusiness problem:\n{business_problem}",
    "api_endpoints": "Design RESTful Flask API endpoints for GET /, GET /health, GET /api/sample-plan, and POST /api/plan. Include request and response details.",
    "uml_component": "Generate only valid PlantUML for a component diagram with Student, Web Interface, Flask API, StudyPlanGenerator, and JSON Study Plan.",
    "uml_sequence": "Generate only valid PlantUML for a sequence diagram showing Student submitting study inputs and the Flask API returning a weekly plan.",
    "website": "Generate a website that displays an automatically generated image and calls POST /api/plan.",
    "tests": "Generate pytest tests for health, sample plan, successful plan, and validation errors.",
}

problem_statement_fallback = """
Students need a reliable way to convert study goals, time constraints, and module difficulty into a structured weekly plan. The proposed system addresses this by generating a personalised study schedule through a Flask API and a web interface.
"""
problem_statement = ai_text(prompts["problem_statement"], problem_statement_fallback, "problem_statement")

personas_fallback = """
## Personas

1. **Student**
   - Responsibilities: Balance coursework, revision, assignments, and independent learning.
   - Needs: A simple planner that converts limited weekly hours into realistic daily tasks.

2. **Module Tutor**
   - Responsibilities: Support students with learning strategies and workload management.
   - Needs: A transparent plan format that can be discussed and adjusted with students.

3. **Academic Advisor**
   - Responsibilities: Help students manage progress and risk across modules.
   - Needs: Clear risk flags when the available time is not enough for the stated learning goal.
"""
personas = ai_text(prompts["personas"], personas_fallback, "personas")

requirements_fallback = """
## Functional Requirements

- The system shall collect course name, difficulty level, available weekly hours, learning goal, and deadline.
- The system shall expose a `POST /api/plan` endpoint that returns a structured weekly study plan as JSON.
- The system shall expose a `GET /health` endpoint for testing and deployment monitoring.
- The website shall display an automatically generated image related to AI study planning.
- The website shall call the Flask API and display the generated plan in a readable format.

## Non-Functional Requirements

- The API shall return appropriate JSON error messages for invalid requests.
- The generated project shall include automated tests for core API behaviour.
- The system shall be deployable using Gunicorn and reproducible using Docker.
- The project shall include CI configuration to run tests on each push.
- The code and documentation shall remain understandable for human review.
"""
requirements = ai_text(prompts["requirements"], requirements_fallback, "requirements")

user_stories = [
    {
        "id": 1,
        "role": "Student",
        "goal": "generate a weekly study plan from my course details",
        "benefit": "I can organise study time more effectively",
        "acceptance_criteria": [
            "Given valid inputs, when I submit the form, then the system returns a weekly plan.",
            "The response includes daily tasks, estimated hours, revision advice, and risk flags.",
        ],
    },
    {
        "id": 2,
        "role": "Student",
        "goal": "see a sample plan before entering my own information",
        "benefit": "I can understand what the system produces",
        "acceptance_criteria": [
            "GET /api/sample-plan returns a valid example plan.",
            "The example contains the same structure as a generated plan.",
        ],
    },
    {
        "id": 3,
        "role": "Developer",
        "goal": "check the health of the deployed API",
        "benefit": "I can verify that deployment is working",
        "acceptance_criteria": [
            "GET /health returns status ok.",
            "The response uses JSON and HTTP 200.",
        ],
    },
    {
        "id": 4,
        "role": "Developer",
        "goal": "run automated tests before deployment",
        "benefit": "I reduce the risk of releasing broken code",
        "acceptance_criteria": [
            "The project includes pytest tests.",
            "GitHub Actions runs the tests on push.",
        ],
    },
]

api_endpoints_fallback = """
## API Endpoints

| Method | Path | Purpose | Success Response |
|---|---|---|---|
| GET | `/` | Serve the website | HTML page |
| GET | `/health` | Health check for tests and deployment | `{ "status": "ok" }` |
| GET | `/api/sample-plan` | Return a demonstration study plan | JSON study plan |
| POST | `/api/plan` | Generate a personalised study plan | JSON study plan |

### POST `/api/plan` Request Body

```json
{
  "course_name": "DTS114TC AI Software Engineering",
  "difficulty": "hard",
  "available_hours": 8,
  "learning_goal": "prepare for coursework and final revision",
  "deadline": "2026-06-07"
}
```
"""
api_endpoints = ai_text(prompts["api_endpoints"], api_endpoints_fallback, "api_endpoints")

write_text(ARTIFACTS / "problem_statement.md", problem_statement)
write_text(ARTIFACTS / "personas.md", personas)
write_text(ARTIFACTS / "requirements.md", requirements)
write_text(ARTIFACTS / "api_endpoints.md", api_endpoints)
write_json(ARTIFACTS / "user_stories.json", {"user_stories": user_stories})

user_stories_md = "## User Stories\n\n" + "\n\n".join(
    f"### Story {story['id']}: {story['role']}\n"
    f"As a {story['role']}, I want to {story['goal']}, so that {story['benefit']}.\n\n"
    "Acceptance criteria:\n" + "\n".join(f"- {item}" for item in story["acceptance_criteria"])
    for story in user_stories
)
write_text(ARTIFACTS / "user_stories.md", user_stories_md)
write_text(ARTIFACTS / "llm_user_story_suggestions.md", ai_text(prompts["user_stories"], user_stories_md, "user_stories_markdown"))

ai_usage_log = (
    "# AI Usage Log\n\n"
    "## Prompts\n\n"
    + "\n".join(f"- **{name}**: {prompt}" for name, prompt in prompts.items())
    + "\n\n## Call Results\n\n"
    + "\n".join(f"- **{item['label']}**: {item['mode']} ({item.get('model') or item.get('reason')})" for item in AI_CALL_LOG)
)
write_text(ARTIFACTS / "ai_usage_log.md", ai_usage_log)

uml_component_fallback = """
@startuml
actor Student
component "Web Interface" as Web
component "Flask API" as API
component "StudyPlanGenerator" as Planner
database "JSON Study Plan" as Plan
Student --> Web : enters study inputs
Web --> API : POST /api/plan
API --> Planner : generate_plan()
Planner --> Plan : creates structured output
Plan --> API : returns plan data
API --> Web : JSON response
Web --> Student : displays weekly plan
@enduml
"""
uml_component = ai_text(prompts["uml_component"], uml_component_fallback, "uml_component")

uml_sequence_fallback = """
@startuml
actor Student
participant "Website" as Website
participant "Flask API" as API
participant "StudyPlanGenerator" as Planner
Student -> Website: Enter course, difficulty, hours, goal, deadline
Website -> API: POST /api/plan
API -> API: Validate request body
API -> Planner: generate_plan(inputs)
Planner --> API: Weekly study plan
API --> Website: JSON response
Website --> Student: Display plan and advice
@enduml
"""
uml_sequence = ai_text(prompts["uml_sequence"], uml_sequence_fallback, "uml_sequence")

write_text(ARTIFACTS / "uml_component.puml", uml_component)
write_text(ARTIFACTS / "uml_sequence.puml", uml_sequence)

def save_placeholder_diagram(path: Path, title: str, labels):
    if Image is None:
        write_text(path.with_suffix(".txt"), f"PIL is not installed. Diagram preview skipped for {title}.")
        return
    img = Image.new("RGB", (1200, 620), "#f8fafc")
    draw = ImageDraw.Draw(img)
    try:
        title_font = ImageFont.truetype("arial.ttf", 34)
        box_font = ImageFont.truetype("arial.ttf", 22)
        small_font = ImageFont.truetype("arial.ttf", 18)
    except Exception:
        title_font = box_font = small_font = ImageFont.load_default()
    draw.rectangle((0, 0, 1200, 86), fill="#1f2937")
    draw.text((40, 24), title, fill="white", font=title_font)
    xs = [60, 300, 555, 830]
    for index, label in enumerate(labels):
        x = xs[index]
        y = 280
        draw.rounded_rectangle((x, y, x + 210, y + 105), radius=12, fill="#e0f2fe", outline="#1f2937", width=2)
        draw.text((x + 20, y + 38), label, fill="#111827", font=box_font)
        if index < len(labels) - 1:
            draw.line((x + 210, y + 52, xs[index + 1], y + 52), fill="#111827", width=4)
            draw.polygon([(xs[index + 1], y + 52), (xs[index + 1] - 12, y + 44), (xs[index + 1] - 12, y + 60)], fill="#111827")
    draw.text((60, 500), "Generated automatically from the notebook's UML prompt and architecture model.", fill="#475569", font=small_font)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)
    print(f"Saved: {path}")

save_placeholder_diagram(ARTIFACTS / "uml_component.png", "Generated UML Component Diagram", ["Student", "Website", "Flask API", "Planner"])
save_placeholder_diagram(ARTIFACTS / "uml_sequence.png", "Generated UML Sequence Diagram Preview", ["Input", "POST", "Generate", "Display"])

def try_generate_api_image(path: Path, prompt: str) -> bool:
    if not has_valid_api_key():
        AI_CALL_LOG.append({"label": "ai_study_image", "mode": "fallback", "reason": "API disabled, missing key, or requests unavailable"})
        return False

    headers = {
        "Authorization": f"Bearer {APIFREE_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": AI_IMAGE_MODEL,
        "num_images": 1,
        "prompt": prompt,
        "quality": AI_IMAGE_QUALITY,
        "size": AI_IMAGE_SIZE,
    }

    try:
        submit = requests.post(f"{APIFREE_BASE_URL}/image/submit", headers=headers, json=payload, timeout=120)
        submit.raise_for_status()
        submit_data = submit.json()
        if submit_data.get("code") != 200:
            raise RuntimeError(submit_data.get("error") or submit_data.get("code_msg") or "image submit failed")
        request_id = submit_data["resp_data"]["request_id"]

        for _ in range(150):
            time.sleep(2)
            check = requests.get(f"{APIFREE_BASE_URL}/image/{request_id}/result", headers=headers, timeout=60)
            check.raise_for_status()
            check_data = check.json()
            if check_data.get("code") != 200:
                raise RuntimeError(check_data.get("code_msg") or "image result check failed")
            resp_data = check_data.get("resp_data") or {}
            status = resp_data.get("status")
            if status in {"success", "completed"}:
                image_url = resp_data["image_list"][0]
                image_response = requests.get(image_url, timeout=120)
                image_response.raise_for_status()
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(image_response.content)
                AI_CALL_LOG.append({"label": "ai_study_image", "mode": "api", "model": AI_IMAGE_MODEL})
                print(f"Saved API-generated image: {path}")
                return True
            if status in {"error", "failed"}:
                raise RuntimeError(resp_data.get("error") or "image generation failed")

        raise TimeoutError("image generation timed out")
    except Exception as exc:
        AI_CALL_LOG.append({"label": "ai_study_image", "mode": "fallback", "reason": str(exc)[:180]})
        return False

def generate_study_image(path: Path):
    image_prompt = (
        "A sharp high-resolution web application hero image for an AI Study Planner dashboard. "
        "Use a 16:9 landscape composition. Show a clean weekly timetable, readable large task cards, "
        "a real-looking line chart for study hours, progress bars for courses, revision timeline nodes, "
        "and a small AI assistant panel. Keep text minimal and large enough to be legible. "
        "Modern academic SaaS dashboard, bright professional interface, crisp edges, no logos."
    )
    if try_generate_api_image(path, image_prompt):
        return

    if Image is None:
        write_text(path.with_suffix(".txt"), "PIL is not installed. Image generation skipped.")
        return
    width, height = 1400, 820
    img = Image.new("RGB", (width, height), "#eef2ff")
    draw = ImageDraw.Draw(img)
    try:
        title_font = ImageFont.truetype("arial.ttf", 54)
        subtitle_font = ImageFont.truetype("arial.ttf", 28)
        card_font = ImageFont.truetype("arial.ttf", 24)
        small_font = ImageFont.truetype("arial.ttf", 18)
    except Exception:
        title_font = subtitle_font = card_font = small_font = ImageFont.load_default()
    for y in range(height):
        ratio = y / height
        draw.line((0, y, width, y), fill=(238 - int(42 * ratio), 242 - int(20 * ratio), 255 - int(18 * ratio)))
    draw.rounded_rectangle((80, 70, 1320, 750), radius=34, fill="#ffffff", outline="#c7d2fe", width=3)
    draw.text((130, 120), "AI Study Planner", fill="#111827", font=title_font)
    draw.text((130, 190), "Generated weekly plan from goals, time, and difficulty", fill="#4b5563", font=subtitle_font)
    cards = [
        (130, 290, 420, 450, "Course", "DTS114TC", "AI Software Engineering", "#dbeafe"),
        (470, 290, 760, 450, "Available Time", "8 hours", "Balanced schedule", "#dcfce7"),
        (810, 290, 1100, 450, "Risk", "Medium", "Revise early", "#fef3c7"),
        (130, 500, 1100, 660, "Weekly Focus", "Requirements -> UML -> API -> Tests -> Deployment", "AI-DLC workflow with human validation", "#ede9fe"),
    ]
    for x1, y1, x2, y2, heading, main, sub, fill in cards:
        draw.rounded_rectangle((x1, y1, x2, y2), radius=22, fill=fill, outline="#9ca3af", width=2)
        draw.text((x1 + 24, y1 + 22), heading, fill="#374151", font=small_font)
        draw.text((x1 + 24, y1 + 58), main, fill="#111827", font=card_font)
        draw.text((x1 + 24, y1 + 102), sub, fill="#4b5563", font=small_font)
    for i, day in enumerate(["Mon", "Tue", "Wed", "Thu", "Fri"]):
        x = 1160
        y = 300 + i * 70
        draw.rounded_rectangle((x, y, x + 90, y + 46), radius=14, fill="#1f2937")
        draw.text((x + 20, y + 13), day, fill="white", font=small_font)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)
    print(f"Saved: {path}")

generate_study_image(STATIC / "ai_study_planner.png")

main_py = r"""
from datetime import datetime
import os
from flask import Flask, jsonify, render_template, request
from flask_cors import CORS


def create_app():
    app = Flask(__name__, template_folder="templates", static_folder="static")
    CORS(app)

    @app.get("/")
    def home():
        return render_template("index.html")

    @app.get("/health")
    def health():
        return jsonify({"status": "ok", "service": "ai-study-planner"}), 200

    @app.get("/api/sample-plan")
    def sample_plan():
        plan = generate_plan(
            course_name="DTS114TC AI Software Engineering",
            difficulty="hard",
            available_hours=8,
            learning_goal="prepare coursework evidence and final revision",
            deadline="2026-06-07",
        )
        return jsonify(plan), 200

    @app.post("/api/plan")
    def create_plan():
        data = request.get_json(silent=True) or {}
        required = ["course_name", "difficulty", "available_hours", "learning_goal", "deadline"]
        missing = [field for field in required if data.get(field) in (None, "")]
        if missing:
            return jsonify({"error": "missing_required_fields", "fields": missing}), 400
        try:
            available_hours = float(data["available_hours"])
        except (TypeError, ValueError):
            return jsonify({"error": "available_hours_must_be_a_number"}), 400
        if available_hours <= 0:
            return jsonify({"error": "available_hours_must_be_positive"}), 400
        difficulty = str(data["difficulty"]).lower().strip()
        if difficulty not in {"easy", "medium", "hard"}:
            return jsonify({"error": "difficulty_must_be_easy_medium_or_hard"}), 400
        plan = generate_plan(
            course_name=str(data["course_name"]).strip(),
            difficulty=difficulty,
            available_hours=available_hours,
            learning_goal=str(data["learning_goal"]).strip(),
            deadline=str(data["deadline"]).strip(),
        )
        return jsonify(plan), 201

    return app


def generate_plan(course_name, difficulty, available_hours, learning_goal, deadline):
    multiplier = {"easy": 0.85, "medium": 1.0, "hard": 1.25}[difficulty]
    effective_hours = round(available_hours * multiplier, 1)
    daily_hours = distribute_hours(effective_hours)
    focus_path = build_focus_path(course_name, learning_goal, difficulty)
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekly_plan = []
    for index, day in enumerate(days):
        topic = focus_path[index % len(focus_path)]
        task_type = "Review and summarise" if index in {0, 3} else "Practice and produce evidence"
        if day == "Sunday":
            task_type = "Reflect, test knowledge, and adjust next week"
        weekly_plan.append({
            "day": day,
            "topic": topic,
            "task": f"{task_type}: {topic}",
            "estimated_hours": daily_hours[index],
        })
    risk_flags = []
    if available_hours < 5 and difficulty in {"medium", "hard"}:
        risk_flags.append("Available hours may be too low for the selected difficulty.")
    if not looks_like_date(deadline):
        risk_flags.append("Deadline format was not recognised; use YYYY-MM-DD for clearer planning.")
    if not risk_flags:
        risk_flags.append("Plan is feasible if tasks are completed consistently.")
    return {
        "course_name": course_name,
        "difficulty": difficulty,
        "available_hours": available_hours,
        "learning_goal": learning_goal,
        "deadline": deadline,
        "weekly_plan": weekly_plan,
        "revision_advice": build_revision_advice(difficulty, available_hours),
        "risk_flags": risk_flags,
        "generated_by": "AI-DLC Study Planner Generator",
    }


def distribute_hours(total_hours):
    weights = [0.15, 0.15, 0.15, 0.15, 0.17, 0.15, 0.08]
    hours = [round(total_hours * weight, 1) for weight in weights]
    hours[-1] = round(hours[-1] + round(total_hours - sum(hours), 1), 1)
    return hours


def build_focus_path(course_name, learning_goal, difficulty):
    base = [
        "Clarify requirements and success criteria",
        "Review core concepts and lecture notes",
        "Create examples or diagrams",
        "Practise implementation tasks",
        "Run tests and fix weak areas",
        "Prepare submission evidence",
        "Reflect and plan improvements",
    ]
    if "software" in course_name.lower() or "api" in learning_goal.lower():
        base[2] = "Model architecture with UML"
        base[3] = "Implement API and website tasks"
        base[4] = "Run pytest and deployment checks"
    if difficulty == "hard":
        base.insert(4, "Reserve extra time for debugging and revision")
    return base


def build_revision_advice(difficulty, available_hours):
    if difficulty == "hard":
        return "Use short daily sessions, test yourself twice, and reserve time for debugging or rework."
    if available_hours >= 8:
        return "Use a balanced plan with concept review, active recall, and practical output."
    return "Prioritise the highest-value topics and keep each task small enough to finish."


def looks_like_date(value):
    try:
        datetime.strptime(value, "%Y-%m-%d")
        return True
    except ValueError:
        return False


app = create_app()

if __name__ == "__main__":
    port = int(os.environ.get("PORT", "5000"))
    app.run(host="0.0.0.0", port=port, debug=False)
"""

write_text(APP / "main.py", main_py)
write_text(APP / "__init__.py", "")

index_html = r"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>AI Study Planner</title>
  <style>
    :root { --ink:#172033; --muted:#5c667a; --line:#d7ddeb; --paper:#fff; --blue:#2457d6; --green:#187861; --bg:#eef3fb; }
    * { box-sizing: border-box; }
    body { margin:0; font-family: Arial, Helvetica, sans-serif; color:var(--ink); background:var(--bg); }
    header { background:#172033; color:white; padding:26px 32px; }
    header h1 { margin:0 0 6px; font-size:30px; letter-spacing:0; }
    header p { margin:0; color:#cbd5e1; }
    main { max-width:1180px; margin:0 auto; padding:28px 20px 48px; display:grid; grid-template-columns:1fr 1fr; gap:24px; }
    .panel { background:var(--paper); border:1px solid var(--line); border-radius:8px; padding:22px; box-shadow:0 12px 28px rgba(23,32,51,.08); }
    .visual { grid-column:1/-1; padding:0; overflow:hidden; }
    .visual img { display:block; width:100%; height:auto; }
    h2 { margin:0 0 16px; font-size:22px; }
    label { display:block; margin:14px 0 6px; font-weight:700; }
    input, select, textarea { width:100%; border:1px solid #b8c2d6; border-radius:6px; padding:11px 12px; font-size:15px; color:var(--ink); background:white; }
    textarea { min-height:96px; resize:vertical; }
    button { margin-top:18px; width:100%; border:0; border-radius:6px; padding:13px 16px; color:white; background:var(--blue); font-size:16px; font-weight:700; cursor:pointer; }
    button.secondary { background:var(--green); }
    button:disabled { opacity:.6; cursor:wait; }
    .meta { color:var(--muted); font-size:14px; line-height:1.55; }
    .plan-list { display:grid; gap:10px; }
    .day-card { border:1px solid var(--line); border-left:5px solid var(--blue); border-radius:7px; padding:12px 14px; background:#fbfdff; }
    .day-card strong { display:block; margin-bottom:4px; }
    .risk { margin-top:14px; padding:12px 14px; border-radius:7px; background:#fff7ed; border:1px solid #fed7aa; color:#7c2d12; }
    .empty { padding:24px; border:1px dashed #aab5ca; border-radius:7px; color:var(--muted); background:#f8fafc; }
    @media (max-width:820px) { main { grid-template-columns:1fr; } }
  </style>
</head>
<body>
  <header>
    <h1>AI Study Planner</h1>
    <p>Generate a structured weekly plan from your course, deadline, study hours, and learning goal.</p>
  </header>
  <main>
    <section class="panel visual">
      <img src="/static/ai_study_planner.png" alt="Generated AI study planning dashboard image">
    </section>
    <section class="panel">
      <h2>Create Plan</h2>
      <form id="plannerForm">
        <label for="course_name">Course name</label>
        <input id="course_name" name="course_name" value="DTS114TC AI Software Engineering" required>
        <label for="difficulty">Difficulty</label>
        <select id="difficulty" name="difficulty" required>
          <option value="easy">Easy</option>
          <option value="medium">Medium</option>
          <option value="hard" selected>Hard</option>
        </select>
        <label for="available_hours">Available weekly hours</label>
        <input id="available_hours" name="available_hours" type="number" min="1" step="0.5" value="8" required>
        <label for="deadline">Deadline</label>
        <input id="deadline" name="deadline" type="date" value="2026-06-07" required>
        <label for="learning_goal">Learning goal</label>
        <textarea id="learning_goal" name="learning_goal" required>Prepare coursework evidence, revise AI-DLC, and verify deployment.</textarea>
        <button id="submitBtn" type="submit">Generate Study Plan</button>
        <button class="secondary" id="sampleBtn" type="button">Load Sample Plan</button>
      </form>
    </section>
    <section class="panel">
      <h2>Generated Plan</h2>
      <div id="summary" class="meta">No plan generated yet.</div>
      <div id="planOutput" class="empty">Submit the form or load a sample plan.</div>
    </section>
  </main>
  <script>
    const form = document.getElementById("plannerForm");
    const sampleBtn = document.getElementById("sampleBtn");
    const submitBtn = document.getElementById("submitBtn");
    const output = document.getElementById("planOutput");
    const summary = document.getElementById("summary");
    function formDataToJson(formElement) { return Object.fromEntries(new FormData(formElement).entries()); }
    function renderPlan(plan) {
      summary.textContent = `${plan.course_name} | ${plan.difficulty} | ${plan.available_hours} hours | deadline: ${plan.deadline}`;
      const days = plan.weekly_plan.map(item => `<div class="day-card"><strong>${item.day}: ${item.topic}</strong><div>${item.task}</div><div class="meta">Estimated time: ${item.estimated_hours} hours</div></div>`).join("");
      const risks = plan.risk_flags.map(flag => `<li>${flag}</li>`).join("");
      output.className = "plan-list";
      output.innerHTML = `${days}<div class="risk"><strong>Revision advice</strong><br>${plan.revision_advice}<ul>${risks}</ul></div>`;
    }
    function renderError(message) { output.className = "empty"; output.textContent = message; }
    form.addEventListener("submit", async event => {
      event.preventDefault();
      submitBtn.disabled = true;
      try {
        const response = await fetch("/api/plan", { method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(formDataToJson(form)) });
        const data = await response.json();
        if (!response.ok) throw new Error(data.error || "Request failed");
        renderPlan(data);
      } catch (error) {
        renderError(`Unable to generate plan: ${error.message}`);
      } finally {
        submitBtn.disabled = false;
      }
    });
    sampleBtn.addEventListener("click", async () => {
      try {
        const response = await fetch("/api/sample-plan");
        renderPlan(await response.json());
      } catch (error) {
        renderError(`Unable to load sample plan: ${error.message}`);
      }
    });
  </script>
</body>
</html>
"""

write_text(TEMPLATES / "index.html", index_html)

test_api_py = r"""
import json
import pytest
from app.main import app


@pytest.fixture()
def client():
    app.config.update(TESTING=True)
    with app.test_client() as test_client:
        yield test_client


def test_health_check(client):
    response = client.get("/health")
    assert response.status_code == 200
    assert response.get_json()["status"] == "ok"


def test_sample_plan(client):
    response = client.get("/api/sample-plan")
    data = response.get_json()
    assert response.status_code == 200
    assert data["course_name"]
    assert len(data["weekly_plan"]) == 7


def test_create_plan_success(client):
    payload = {
        "course_name": "DTS114TC AI Software Engineering",
        "difficulty": "hard",
        "available_hours": 8,
        "learning_goal": "prepare coursework and deployment evidence",
        "deadline": "2026-06-07",
    }
    response = client.post("/api/plan", data=json.dumps(payload), content_type="application/json")
    data = response.get_json()
    assert response.status_code == 201
    assert data["difficulty"] == "hard"
    assert len(data["weekly_plan"]) == 7
    assert "revision_advice" in data


def test_create_plan_validation_error(client):
    response = client.post("/api/plan", json={"course_name": "DTS114TC"})
    data = response.get_json()
    assert response.status_code == 400
    assert data["error"] == "missing_required_fields"
"""

write_text(TESTS / "test_api.py", test_api_py)

write_text(BASE / "requirements.txt", """
Flask
flask-cors
gunicorn
pytest==8.4.2
Pillow
""")

write_text(BASE / "README.md", """
# AI Study Planner

This project was generated by the `AI-DLC Study Planner Generator` notebook. It contains a Flask API, website, generated image, SDLC artefacts, UML diagrams, tests, Docker configuration, and GitHub Actions workflow.

## Run Locally

```bash
pip install -r requirements.txt
python app/main.py
```

Open `http://127.0.0.1:5000`.

## Test

```bash
pytest -q -p no:cacheprovider
```

## Deploy

Recommended Render settings:

- Build command: `pip install -r requirements.txt`
- Start command: `gunicorn app.main:app`
""")

write_text(BASE / "Dockerfile", """
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 5000
CMD ["gunicorn", "app.main:app", "--bind", "0.0.0.0:5000"]
""")

write_text(BASE / ".dockerignore", """
__pycache__/
*.pyc
.pytest_cache/
.git/
.env
.venv/
venv/
""")

write_text(BASE / "run_local_server.bat", """
@echo off
cd /d "%~dp0"
call conda activate ai_in_se_chapter_04
python app\\main.py
pause
""")

write_text(BASE / "open_local_site.bat", """
@echo off
start "" "http://127.0.0.1:5000"
""")

write_text(WORKFLOWS / "ci.yml", """
name: CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run tests
        run: pytest -q -p no:cacheprovider
""")

final_ai_usage_log = (
    "# AI Usage Log\n\n"
    "This file records the AI-specific tooling used by the notebook. "
    "API keys are loaded from `.env` and are never written into generated artefacts.\n\n"
    f"- Provider: {AI_PROVIDER}\n"
    f"- Text model: {AI_TEXT_MODEL}\n"
    f"- Image model: {AI_IMAGE_MODEL}\n\n"
    "## Prompts\n\n"
    + "\n".join(f"- **{name}**: {prompt}" for name, prompt in prompts.items())
    + "\n\n## Call Results\n\n"
    + "\n".join(f"- **{item['label']}**: {item['mode']} ({item.get('model') or item.get('reason')})" for item in AI_CALL_LOG)
)
write_text(ARTIFACTS / "ai_usage_log.md", final_ai_usage_log)

print("Generation complete.")
print("Next validation commands:")
print("cd generated_project")
print("pip install -r requirements.txt")
print("pytest -q -p no:cacheprovider")
print("python app/main.py")
print("Open http://127.0.0.1:5000")

## 2. Validation

After running the code cell, open a terminal in `Task1/generated_project` and run:

```bash
pip install -r requirements.txt
pytest
python app/main.py
```

Then open `http://127.0.0.1:5000`.